In [0]:
SELECT * FROM workspace.default.gold_resumo_por_dia;

In [0]:
-- 1. CAMADA BRONZE
CREATE TABLE IF NOT EXISTS gorjetas_bronze AS
SELECT * FROM workspace.default.tips;

-- 2. CAMADA SILVER
CREATE TABLE IF NOT EXISTS gorjetas_silver AS
SELECT 
    CAST(total_bill AS DOUBLE) AS valor_conta,
    CAST(tip AS DOUBLE) AS valor_gorjeta,
    CASE WHEN sex = 'Female' THEN 'Feminino' ELSE 'Masculino' END AS genero_pagador,
    CASE WHEN smoker = 'Yes' THEN 'Sim' ELSE 'Não' END AS cliente_fumante,
    CASE 
        WHEN day = 'Thur' THEN 'Quinta'
        WHEN day = 'Fri' THEN 'Sexta'
        WHEN day = 'Sat' THEN 'Sábado'
        WHEN day = 'Sun' THEN 'Domingo'
        ELSE day 
    END AS dia_semana,
    CASE WHEN time = 'Lunch' THEN 'Almoço' ELSE 'Jantar' END AS periodo,
    CAST(size AS INT) AS qtde_pessoas
FROM gorjetas_bronze;

-- 3. CAMADA GOLD

-- Pergunta 1: Valor médio da conta e da gorjeta por dia da semana
CREATE TABLE IF NOT EXISTS gold_resumo_por_dia AS
SELECT 
    dia_semana,
    COUNT(*) AS total_mesas,
    ROUND(AVG(valor_conta), 2) AS conta_media,
    ROUND(AVG(valor_gorjeta), 2) AS gorjeta_media
FROM gorjetas_silver
GROUP BY dia_semana
ORDER BY conta_media DESC;

-- Pergunta 2: Gorjeta por perfil (Fumante vs Não Fumante)
CREATE TABLE IF NOT EXISTS gold_resumo_fumantes AS
SELECT 
    cliente_fumante,
    COUNT(*) AS total_mesas,
    ROUND(AVG(valor_gorjeta), 2) AS gorjeta_media,
    ROUND(AVG((valor_gorjeta / valor_conta) * 100), 2) AS pct_medio_gorjeta
FROM gorjetas_silver
GROUP BY cliente_fumante;